In [9]:
import requests
from pathlib import Path
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_ollama import ChatOllama
from bs4 import BeautifulSoup
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [5]:
# NASA Source
NASA_URL = "https://science.nasa.gov/solar-system/planets/"

response = requests.get(
    NASA_URL,
    timeout = 20
)

response.raise_for_status()

print("NASA page fetched successfully!")
print("Status code:", response.status_code)
print("Characters:", len(response.text))

NASA page fetched successfully!
Status code: 200
Characters: 318616


In [21]:
NASA_URLS = {
    "planets": "https://science.nasa.gov/solar-system/planets/",
    "jupiter": "https://science.nasa.gov/jupiter/jupiter-facts/",
    "solar_system": "https://science.nasa.gov/solar-system/solar-system-facts/",
    "planet_formation": "https://science.nasa.gov/exoplanets/how-do-planets-form/",
}

for name, url in NASA_URLS.items():
    print(f"{name}: {url}")

planets: https://science.nasa.gov/solar-system/planets/
jupiter: https://science.nasa.gov/jupiter/jupiter-facts/
solar_system: https://science.nasa.gov/solar-system/solar-system-facts/
planet_formation: https://science.nasa.gov/exoplanets/how-do-planets-form/


In [22]:
responses = {}

for name, url in NASA_URLS.items():

    response = requests.get(
        url,
        timeout=20
    )

    response.raise_for_status()

    responses[name] = response.text

    print(
        f"{name:<20} "
        f"Status: {response.status_code} | "
        f"Characters: {len(response.text)}"
    )

planets              Status: 200 | Characters: 318615
jupiter              Status: 200 | Characters: 277371
solar_system         Status: 200 | Characters: 313350
planet_formation     Status: 200 | Characters: 270018


In [32]:
documents = []

for name, html in responses.items():

    soup = BeautifulSoup(html, "html.parser")

    # Remove obvious non-content elements
    for element in soup([
        "script",
        "style",
        "nav",
        "footer",
        "header",
        "aside",
        "form"
    ]):
        element.decompose()

    # Try to locate the main article/content area
    main_content = (
        soup.find("main")
        or soup.find("article")
        or soup.body
    )

    text = main_content.get_text(
        separator="\n",
        strip=True
    )

    documents.append(
        Document(
            page_content=text,
            metadata={
                "source": "NASA Science",
                "topic": name,
                "url": NASA_URLS[name]
            }
        )
    )

    print(
        f"{name:<20} "
        f"{len(text):,} characters"
    )

planets              3,963 characters
jupiter              11,934 characters
solar_system         7,198 characters
planet_formation     3,161 characters


In [57]:
for doc in documents:
    print("=" * 60)
    print("TOPIC:", doc.metadata["topic"])
    print("=" * 60)
    print(doc.page_content[:1500])

TOPIC: planets
Explore This Section
About the Planets
Our  solar system has eight planets: Mercury, Venus, Earth, Mars, Jupiter, Saturn, Uranus, and Neptune. There are five officially recognized dwarf planets in our solar system: Ceres, Pluto, Haumea, Makemake, and Eris.
8
Planets
5
Dwarf Planets
Introduction
What is a planet? The word traces back to the ancient Greek word planēt, which means “wanderer.” A more modern definition can be found in the Merriam-Webster dictionary, which defines a planet as “any of the large bodies that revolve around the Sun in the solar system.”
In 2006, the International Astronomical Union (IAU) — a group of astronomers that names objects in our solar system — agreed on their own definition of
“planet.”
This new definition  caused Pluto's famous “demotion” to a dwarf planet, and reduced the solar system census from nine planets to eight.
Learn More: ‘What is a planet?’
Inner Planets
The first four planets from the Sun are Mercury, Venus, Earth, and Mars. 

In [33]:
nasa_doc = Document(
    page_content = nasa_text,
    metadata = {
        "source": "NASA Science",
        "url": NASA_URL,
        "topic": "Solar System Planets"
    }
)

print("Document created!")
print("Source:", nasa_doc.metadata["source"])
print("URL:", nasa_doc.metadata["url"])

Document created!
Source: NASA Science
URL: https://science.nasa.gov/solar-system/planets/


In [38]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150
)

chunks = splitter.split_documents(documents)

print("Documents:", len(documents))
print("Chunks:", len(chunks))

Documents: 4
Chunks: 33


In [42]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-l6-v2"
)

print("Embedding model loaded!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded!


In [56]:
vectorstore = FAISS.from_documents(
    chunks,
    embeddings
)

retriever = vectorstore.as_retriever(
    search_kwargs={"k": 4}
)

print("FAISS rebuilt!")

FAISS rebuilt!


In [55]:
query = "Why is Jupiter the largest planet in the solar system?"

results = retriever.invoke(query)

for i, doc in enumerate(results, start=1):
    print("=" * 70)
    print(f"RESULT {i}")
    print("=" * 70)
    print("TOPIC:", doc.metadata["topic"])
    print()
    print(doc.page_content[:1000])

RESULT 1
TOPIC: planets

Jupiter Facts
Jupiter is the fifth planet from the Sun, and the largest planet in our solar system.
Explore Jupiter
Saturn Facts
Saturn is the sixth planet from the Sun, the second largest planet in our solar system.
Explore Saturn
Uranus Facts
Uranus is the seventh planet from the Sun, and the third largest planet in our solar system.
Explore Uranus
Neptune Facts
Neptune is the eighth and most distant planet in our solar system. It's the fourth largest planet.
Explore Neptune
Dwarf Planets
Dwarf planets include longtime favorite Pluto, Ceres, Makemake, Haumea, and Eris. Ceres is the only dwarf planet in the inner solar system. It's in the main asteroid belt between Mars and Jupiter.
Ceres Facts
Dwarf planet Ceres is the largest object in the asteroid belt between Mars and Jupiter, and it's the only dwarf planet in the inner solar system.
Explore Ceres
Pluto Facts
RESULT 2
TOPIC: jupiter

Namesake
Jupiter, being the biggest planet, gets its name from the king o

In [54]:
for i, chunk in enumerate(chunks[:5]):
    print("=" * 60)
    print(f"Chunk {i+1}")
    print("=" * 60)
    print(chunk.page_content[:500])

Chunk 1
Explore This Section
About the Planets
Our  solar system has eight planets: Mercury, Venus, Earth, Mars, Jupiter, Saturn, Uranus, and Neptune. There are five officially recognized dwarf planets in our solar system: Ceres, Pluto, Haumea, Makemake, and Eris.
8
Planets
5
Dwarf Planets
Introduction
What is a planet? The word traces back to the ancient Greek word planēt, which means “wanderer.” A more modern definition can be found in the Merriam-Webster dictionary, which defines a planet as “any of 
Chunk 2
Learn More: ‘What is a planet?’
Inner Planets
The first four planets from the Sun are Mercury, Venus, Earth, and Mars.  These inner planets also are known as terrestrial planets because they have solid surfaces.
Mercury Facts
Mercury is the planet nearest to the Sun, and the smallest planet in our solar system.
Explore Mercury
Venus Facts
Venus is the second planet from the Sun, and the sixth largest planet.
Explore Venus
Earth Facts
Earth – our home planet – is the third planet

In [43]:
retriever = vectorstore.as_retriever(
    search_kwargs={
        "k": 4
    }
)

print("Retriever ready!")

Retriever ready!


In [49]:
query = "How did Jupiter form and acquire its enormous mass?"

results = retriever.invoke(query)

for i, doc in enumerate(results, start=1):

    print("\n" + "=" * 80)
    print(f"RESULT {i}")
    print("=" * 80)

    print("TOPIC :", doc.metadata["topic"])
    print("SOURCE:", doc.metadata["source"])
    print()
    print(doc.page_content)


RESULT 1
TOPIC : jupiter
SOURCE: NASA Science

NASA's Europa Clipper
mission slated to launch in 2024.
›
More on Jupiter's Moons
Rings
Discovered in 1979 by NASA's Voyager 1 spacecraft, Jupiter's rings were a surprise. The rings are composed of small, dark particles, and they are difficult to see except when backlit by the Sun. Data from the Galileo spacecraft indicate that Jupiter's ring system may be formed by dust kicked up as interplanetary meteoroids smash into the giant planet's small innermost moons.
Formation
Jupiter took shape along with rest of the solar system about 4.6 billion years ago. Gravity pulled swirling gas and dust together to form this gas giant. Jupiter took most of the mass left over after the formation of the Sun, ending up with more than twice the combined material of the other bodies in the solar system. In fact, Jupiter has the same ingredients as a star, but it did not grow massive enough to ignite.

RESULT 2
TOPIC : jupiter
SOURCE: NASA Science

About 4 b

In [45]:
llm = ChatOllama(model="llama3.2")
response = llm.invoke("What is Jupiter?")
print(response.content)

Jupiter is the fifth planet from the Sun in our solar system. It is a gas giant, meaning that it is primarily composed of hydrogen and helium gases. Jupiter is the largest planet in our solar system, with a diameter of approximately 142,984 kilometers (88,846 miles).

Here are some interesting facts about Jupiter:

1. **Massive size**: Jupiter is so large that it could fit over 1,300 Earths inside of it.
2. **Stormy weather**: Jupiter's atmosphere is home to some incredible storms, including the Great Red Spot, a persistent anticyclonic storm that has been raging for centuries.
3. **Magnetic field**: Jupiter has an incredibly strong magnetic field, which is powered by its rapid rotation and convection in the planet's interior.
4. **Moons**: Jupiter has a system of 92 confirmed moons, with four of them (Io, Europa, Ganymede, and Callisto) being some of the largest in the solar system.
5. **Atmospheric features**: Jupiter's atmosphere is characterized by strong winds, lightning storms, a

In [50]:
context = "\n\n".join(
    doc.page_content
    for doc in results
)

prompt = f"""
You are CosmoLens, an AI astronomy assistant.

Answer the user's question using ONLY the provided NASA context.

Do not add facts that are not supported by the context.

If the context does not contain enough information,
say that the available NASA context does not provide
enough information.

NASA CONTEXT:
{context}

USER QUESTION:
{query}

Give a clear and concise scientific explanation.
"""

response = llm.invoke(prompt)

print(response.content)

According to the provided NASA context, Jupiter formed along with the rest of the solar system about 4.6 billion years ago. Gravity pulled swirling gas and dust together to form this gas giant, with Jupiter taking most of the mass left over after the formation of the Sun. This process resulted in Jupiter having more than twice the combined material of the other bodies in the solar system, making it a massive planet with a huge amount of mass.


In [52]:
response = llm.invoke(prompt)
print(response.content)

According to NASA's context, Jupiter formed along with the rest of the solar system approximately 4.6 billion years ago. Gravity pulled swirling gas and dust together, causing this gas giant to take shape. As a result, Jupiter acquired its enormous mass by capturing most of the material left over after the formation of the Sun. This process occurred before Jupiter settled into its current position in the outer solar system, around 4 billion years ago.
